# Chapter 15: GITHUB_TOKEN, Permissions and Least Privilege (notebook edition)

## Learning Objectives

- Predict the permission matrix
- Read a 403 and its header
- Explain what no setting can grant

In [ ]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

GHA_MODE = os.environ.get("GHA_MODE", "fixture")  # "fixture" | "live"
GHA_REPO = os.environ.get("GHA_REPO", "Friend09/practice_git_intro_to_github_actions")
print("mode:", GHA_MODE, "| repo:", GHA_REPO)

## 1. The measured matrix

Six jobs, three API calls each. A tag needs `contents: write`, a label needs `issues: write`, and listing secrets is not grantable.

In [ ]:
import json
s = json.loads((ROOT/"fixtures"/"token_ch15_summary.json").read_text())
print(f'{"job":15} tag label secrets')
for job, row in s["matrix"].items():
    print(f'{job:15} {row["tag"]}  {row["label"]}   {row["secrets"]}')
assert s["matrix"]["issues_write"] == {"tag": 403, "label": 201, "secrets": 403}

## 2. Predict it with the permissions model

Unlisted scopes become `none`; a job block replaces the workflow block; `;` in the header means 'or'.

In [ ]:
from intro_gha.perms import effective, satisfies
JOBS = {"none": {}, "contents_read": {"contents": "read"}, "contents_write": {"contents": "write"},
        "issues_write": {"issues": "write"}, "both_write": {"contents": "write", "issues": "write"}, "write_all": "write-all"}
wrong = 0
for job, block in JOBS.items():
    perms = effective({"contents": "read"}, block)
    for probe, header in s["accepted_permissions"].items():
        wrong += satisfies(perms, header) != (s["matrix"][job][probe] in (200, 201))
print("mismatches:", wrong)
assert wrong == 0

## 3. The default and the 403

A workflow with no `permissions:` key got 403 on every write; the error and the header explain why.

In [ ]:
print(s["default_workflow_run"], s["repo_settings"]["default_workflow_permissions"])
print(s["error_message"]); print(s["accepted_permissions"])
assert s["default_workflow_run"] == {"tag": 403, "label": 403, "secrets": 403}

## 4. What no setting can grant

`write-all` cannot list secrets: the needed `secrets=read` is not a workflow-token scope.

In [ ]:
from intro_gha.perms import SCOPES
print("secrets" in SCOPES, satisfies(effective(None, "write-all"), "secrets=read"))
assert "secrets" not in SCOPES and not satisfies(effective(None, "write-all"), "secrets=read")

## 5. The rate limit, measured

Docs said 1,000 an hour; 1,150 calls all succeeded and the header said 5,000.

In [ ]:
rl = s["rate_limit"]
print(rl)
print("counter drop:", rl["remaining_before"] - rl["remaining_after"])
assert rl["calls_ok"] == 1150 > rl["docs_limit"] and rl["remaining_before"] - rl["remaining_after"] == 1069

## Takeaways & Next Steps

- Set one scope and the rest become `none`.
- Read `X-Accepted-GitHub-Permissions` to fix a 403.
- Measure rate limits yourself.
- Next: Chapter 16, supply-chain security.

## Chapter Link

Read: `learning_modules/chapter_15_token_and_permissions.md`